# Yankees Attendance Forecasting
Ian Kimura | UCLA STATS 415

Portable copy of the original Colab analysis. Run from Jupyter after placing the four season CSVs in `data/raw/`. See the README for evaluation limitations. The final future-forecast section is an illustrative legacy scenario, not a current operational forecast.

Loading the Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('default')
import seaborn as sns
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf
from scipy.signal import periodogram
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error


Loading the Data

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data" / "raw"
assert DATA.is_dir(), "Start Jupyter from the project root or notebooks directory."

In [ ]:
df2022 = pd.read_csv(DATA / "yankees_2022.csv")
df2023 = pd.read_csv(DATA / "yankees_2023.csv")
df2024 = pd.read_csv(DATA / "yankees_2024.csv")
df2025 = pd.read_csv(DATA / "yankees_2025.csv")

In [ ]:
print(df2022.columns)

In [ ]:
df2022.head()

Data Cleaning Pipeline

In [ ]:
def clean_yankees_season(df, season):
    df = df.dropna(how="all").copy()

    # keep home games only
    df_home = df[df["Unnamed: 4"].isna()].copy()

    # clean attendance
    df_home["Attendance"] = pd.to_numeric(df_home["Attendance"], errors="coerce")
    df_home = df_home.dropna(subset=["Attendance"])
    df_home["Attendance"] = df_home["Attendance"].astype(int)

    # remove doubleheader marker like "(2)"
    df_home["Date"] = (
        df_home["Date"]
        .astype(str)
        .str.replace(r"\s\(\d+\)", "", regex=True)
    )

    # add year and convert date
    df_home["Date"] = pd.to_datetime(
        df_home["Date"] + f" {season}",
        format="%A %b %d %Y"
    )

    # features
    df_home["DayOfWeek"] = df_home["Date"].dt.day_name()
    df_home["Season"] = season

    # keep only needed columns
    df_home = df_home[["Date", "Attendance", "DayOfWeek", "Season"]]


    # reset index
    df_home = df_home.reset_index(drop=True)

    return df_home

In [ ]:
df2022_home = clean_yankees_season(df2022, 2022)
df2023_home = clean_yankees_season(df2023, 2023)
df2024_home = clean_yankees_season(df2024, 2024)
df2025_home = clean_yankees_season(df2025, 2025)

In [ ]:
print("2022:", len(df2022_home))
print("2023:", len(df2023_home))
print("2024:", len(df2024_home))
print("2025:", len(df2025_home))

In [ ]:
yankees_df = pd.concat(
    [df2022_home, df2023_home, df2024_home, df2025_home],
    ignore_index=True
)

yankees_df["t"] = np.arange(1, len(yankees_df) + 1)

In [ ]:
yankees_df.head()

In [ ]:
yankees_df.tail(15)

In [ ]:
plt.figure(figsize=(16,6))

plt.plot(
    yankees_df["t"],
    yankees_df["Attendance"],
    color="black",
    linewidth=1
)
plt.title("New York Yankees Home Attendance (2022-2025)")
plt.xlabel("Event Time (Home Games)")
plt.ylabel("Attendance")

plt.show()

Split the dataset

In [ ]:
train_df = yankees_df[yankees_df["Season"] != 2025].copy()
test_df  = yankees_df[yankees_df["Season"] == 2025].copy()

train_ts = train_df["Attendance"].values
test_ts = test_df["Attendance"].values

In [ ]:
print(len(train_ts))
print(len(test_ts))

Plot ACF and PACF

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10,8))

# approximate confidence interval
conf = 1.96 / np.sqrt(len(train_ts))

# ACF
plot_acf(
    train_ts,
    lags=40,
    ax=axes[0],
    alpha=None,
    color="black",
    vlines_kwargs={"colors":"black","linewidth":1},
    marker=""
)

axes[0].axhline(conf, linestyle='--', color='blue', linewidth=1)
axes[0].axhline(-conf, linestyle='--', color='blue', linewidth=1)

axes[0].set_title("ACF of Yankees Attendance")
axes[0].set_xlabel("Lag")
axes[0].set_ylabel("ACF")

# PACF
plot_pacf(
    train_ts,
    lags=40,
    method='ywm',
    ax=axes[1],
    alpha=None,
    color="black",
    vlines_kwargs={"colors":"black","linewidth":1},
    marker=""
)

axes[1].axhline(conf, linestyle='--', color='blue', linewidth=1)
axes[1].axhline(-conf, linestyle='--', color='blue', linewidth=1)

axes[1].set_title("PACF of Yankees Attendance")
axes[1].set_xlabel("Lag")
axes[1].set_ylabel("Partial ACF")

plt.tight_layout()
plt.show()

Periodogram

In [ ]:

freqs, power = periodogram(train_ts)

plt.figure(figsize=(12,6))

plt.plot(
    freqs,
    power,
    color="black",
    linewidth=1
)

plt.title("Periodogram of Yankees Attendance")
plt.xlabel("Frequency")
plt.ylabel("Spectral Density")

plt.show()

# Phase 2: Deterministic Decomposition

In [ ]:
# LOWESS trend fit
lowess_fit = lowess(
    endog=train_df["Attendance"],
    exog=train_df["t"],
    frac=0.15
)

# extract fitted trend values
trend = lowess_fit[:, 1]

In [ ]:
plt.figure(figsize=(16,6))

# raw series
plt.plot(
    train_df["t"],
    train_df["Attendance"],
    color="black",
    linewidth=1,
    label="Raw Attendance"
)

# smooth trend
plt.plot(
    train_df["t"],
    trend,
    color="red",
    linewidth=2,
    label="LOWESS Trend"
)

plt.title("LOWESS Trend Fit for Yankees Attendance")
plt.xlabel("Event Time")
plt.ylabel("Attendance")

plt.legend()

plt.show()

In [ ]:
train_df["Detrended"] = (
    train_df["Attendance"] - trend
)
plt.figure(figsize=(16,6))

plt.plot(
    train_df["t"],
    train_df["Detrended"],
    color="black",
    linewidth=1
)

plt.axhline(
    y=0,
    color="red",
    linestyle="--"
)

plt.title("Detrended Yankees Attendance")
plt.xlabel("Event Time")
plt.ylabel("Detrended Attendance")

plt.show()

Weekly Cycle Removal

In [ ]:
weekday_means = (
    train_df
    .groupby("DayOfWeek")["Detrended"]
    .mean()
)

weekday_means

In [ ]:
train_df["CycleAdjusted"] = (
    train_df["Detrended"]
    - train_df["DayOfWeek"].map(weekday_means)
)

plt.figure(figsize=(16,6))

plt.plot(
    train_df["t"],
    train_df["CycleAdjusted"],
    color="black",
    linewidth=1
)

plt.axhline(
    y=0,
    color="red",
    linestyle="--"
)

plt.title("Trend and Weekly Cycle Adjusted Attendance")
plt.xlabel("Event Time")
plt.ylabel("Residual Attendance")

plt.show()

Residual Diagnostics

In [ ]:
residual_ts = train_df["CycleAdjusted"].values

fig, axes = plt.subplots(2, 1, figsize=(10,8))

conf = 1.96 / np.sqrt(len(residual_ts))

# ACF
plot_acf(
    residual_ts,
    lags=40,
    ax=axes[0],
    alpha=None,
    color="black",
    vlines_kwargs={"colors":"black","linewidth":1},
    marker=""
)

axes[0].axhline(conf, linestyle='--', color='blue', linewidth=1)
axes[0].axhline(-conf, linestyle='--', color='blue', linewidth=1)

axes[0].set_title("ACF of Residual Attendance")
axes[0].set_xlabel("Lag")
axes[0].set_ylabel("ACF")

# PACF
plot_pacf(
    residual_ts,
    lags=40,
    method='ywm',
    ax=axes[1],
    alpha=None,
    color="black",
    vlines_kwargs={"colors":"black","linewidth":1},
    marker=""
)

axes[1].axhline(conf, linestyle='--', color='blue', linewidth=1)
axes[1].axhline(-conf, linestyle='--', color='blue', linewidth=1)

axes[1].set_title("PACF of Residual Attendance")
axes[1].set_xlabel("Lag")
axes[1].set_ylabel("Partial ACF")

plt.tight_layout()
plt.show()

# Phase 3: Stochastic Modeling and Forecasting

In [ ]:
# Candidate ARMA models to compare
candidate_orders = [
    (1,0,0),  # AR(1)
    (2,0,0),  # AR(2)
    (3,0,0),  # AR(3)

    (0,0,1),  # MA(1)
    (0,0,2),  # MA(2)
    (0,0,3),  # MA(3)

    (1,0,1),  # ARMA(1,1)
    (1,0,2),  # ARMA(1,2)
    (2,0,1),  # ARMA(2,1)
    (2,0,2),  # ARMA(2,2)
]

results = []

for order in candidate_orders:
    p, d, q = order

    try:
        model = ARIMA(residual_ts, order=order).fit()

        if p > 0 and q > 0:
            model_name = f"ARMA({p},{q})"
        elif p > 0:
            model_name = f"AR({p})"
        else:
            model_name = f"MA({q})"

        results.append({
            "Model": model_name,
            "Order": order,
            "AIC": model.aic,
            "BIC": model.bic,
            "LogLik": model.llf
        })

    except Exception as e:
        print(f"Model {order} failed: {e}")

model_comparison = pd.DataFrame(results)

model_comparison["AIC"] = model_comparison["AIC"].round(2)
model_comparison["BIC"] = model_comparison["BIC"].round(2)
model_comparison["LogLik"] = model_comparison["LogLik"].round(2)

model_comparison = model_comparison.sort_values("AIC").reset_index(drop=True)

model_comparison

In [ ]:
best_model = ARIMA(
    residual_ts,
    order=(2,0,1)
).fit()

print(best_model.summary())

In [ ]:
ma1_model = ARIMA(
    residual_ts,
    order=(0,0,1)
).fit()

print(ma1_model.summary())

In [ ]:
def forecast_attendance(model, test_df, trend, weekday_means):
    forecast_steps = len(test_df)

    # 1. Forecast residual component
    residual_forecast = model.forecast(steps=forecast_steps)

    # 2. Extend trend using last LOWESS trend value
    trend_forecast = np.repeat(trend[-1], forecast_steps)

    # 3. Add weekday cycle back
    test_cycle = test_df["DayOfWeek"].map(weekday_means).values

    # 4. Reconstruct attendance forecast
    final_forecast = trend_forecast + test_cycle + residual_forecast

    return final_forecast


# Forecasts
ma1_forecast = forecast_attendance(ma1_model, test_df, trend, weekday_means)
arma21_forecast = forecast_attendance(best_model, test_df, trend, weekday_means)

# RMSE
ma1_rmse = np.sqrt(mean_squared_error(test_df["Attendance"], ma1_forecast))
arma21_rmse = np.sqrt(mean_squared_error(test_df["Attendance"], arma21_forecast))

rmse_comparison = pd.DataFrame({
    "Model": ["MA(1)", "ARMA(2,1)"],
    "AIC": [ma1_model.aic, best_model.aic],
    "BIC": [ma1_model.bic, best_model.bic],
    "RMSE": [ma1_rmse, arma21_rmse]
})

rmse_comparison[["AIC", "BIC", "RMSE"]] = rmse_comparison[["AIC", "BIC", "RMSE"]].round(2)

rmse_comparison

Residual Diagnostics of Final Model
- verifying if MA(1) residuals are approximately white noise

In [ ]:
final_residuals = ma1_model.resid

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10,8))

conf = 1.96 / np.sqrt(len(final_residuals))

# ACF
plot_acf(
    final_residuals,
    lags=40,
    ax=axes[0],
    alpha=None,
    color="black",
    vlines_kwargs={"colors":"black","linewidth":1},
    marker=""
)

axes[0].axhline(conf, linestyle='--', color='blue', linewidth=1)
axes[0].axhline(-conf, linestyle='--', color='blue', linewidth=1)

axes[0].set_title("ACF of MA(1) Residuals")

# PACF
plot_pacf(
    final_residuals,
    lags=40,
    method='ywm',
    ax=axes[1],
    alpha=None,
    color="black",
    vlines_kwargs={"colors":"black","linewidth":1},
    marker=""
)

axes[1].axhline(conf, linestyle='--', color='blue', linewidth=1)
axes[1].axhline(-conf, linestyle='--', color='blue', linewidth=1)

axes[1].set_title("PACF of MA(1) Residuals")

plt.tight_layout()
plt.show()

In [ ]:
final_residuals = ma1_model.resid

In [ ]:
freqs_resid, power_resid = periodogram(final_residuals)

plt.figure(figsize=(12,6))

plt.plot(
    freqs_resid,
    power_resid,
    color="black",
    linewidth=1
)

plt.title("Periodogram of MA(1) Residuals")
plt.xlabel("Frequency")
plt.ylabel("Spectral Density")

plt.show()

Forecasting on test set

In [ ]:
forecast_steps = len(test_df)

residual_forecast = ma1_model.forecast(
    steps=forecast_steps
)

print(residual_forecast[:5])

In [ ]:
last_trend_value = trend[-1]

trend_forecast = np.repeat(
    last_trend_value,
    forecast_steps
)

In [ ]:
test_cycle = test_df["DayOfWeek"].map(
    weekday_means
).values

In [ ]:
final_forecast = (
    trend_forecast
    + test_cycle
    + residual_forecast
)

In [ ]:
from sklearn.metrics import mean_squared_error

rmse = np.sqrt(
    mean_squared_error(
        test_df["Attendance"],
        final_forecast
    )
)

print("RMSE:", rmse)

In [ ]:
plt.figure(figsize=(16,6))

# actual
plt.plot(
    test_df["t"],
    test_df["Attendance"],
    color="black",
    linewidth=1,
    label="Actual Attendance"
)

# forecast
plt.plot(
    test_df["t"],
    final_forecast,
    color="red",
    linewidth=2,
    label="Forecast"
)

plt.title("Forecast vs Actual Yankees Attendance (2025)")
plt.xlabel("Event Time")
plt.ylabel("Attendance")

plt.legend()

plt.show()

Future forecasts

In [ ]:
future_steps = 20

future_residuals = ma1_model.forecast(
    steps=future_steps
)

In [ ]:
future_days = (
    list(test_df["DayOfWeek"][-7:])
    * 3
)[:future_steps]

In [ ]:
#weekday effects
future_cycle = pd.Series(future_days).map(
    weekday_means
).values

In [ ]:
future_trend = np.repeat(
    last_trend_value,
    future_steps
)

In [ ]:
future_forecast = (
    future_trend
    + future_cycle
    + future_residuals
)

In [ ]:
future_t = np.arange(
    yankees_df["t"].max() + 1,
    yankees_df["t"].max() + future_steps + 1
)

In [ ]:
plt.figure(figsize=(16,6))

# historical
plt.plot(
    yankees_df["t"],
    yankees_df["Attendance"],
    color="black",
    linewidth=1,
    label="Historical Attendance"
)

# future forecast
plt.plot(
    future_t,
    future_forecast,
    color="red",
    linewidth=2,
    label="Future Forecast"
)

plt.title("Future Yankees Attendance Forecast")
plt.xlabel("Event Time")
plt.ylabel("Attendance")

plt.legend()

plt.show()